# Linear quadratic regulator
This is my first time encountering mister LQR! Had to do quite a bit of slow reading / thinking to understand what the heck is going on. Had a helpful chat with monsieur [Claude](https://claude.ai/share/bde4ff40-30f0-4a4d-9417-74c95b96283e). My current understanding is below.

### Deriving the equations of motion, or: "given where the system is and what force I'm applying, how are the cart and the pole accelerating right now?"
For this part, I'll be walking through [these UCSB lecture notes](https://courses.ece.ucsb.edu/ECE594/594D_W10Byl/hw/cartpole_eom.pdf) to understand the derivation of the cartpole equations of motion. I'll reference equations in these notes below so it might be handy to open the link.

First, we take the equations of motion of the (frictionless) cartpole system, and stick them into the Lagrangian formulation (here we refer to Lagrangian *mechanics*, not to be confused with the Lagrangian *multiplier* for constrained optimization), $L = T^* - V$, where $T^*$ is kinetic energy and $V$ is potential energy. This is essentially an alternative way to view the physics of the system from the Newtonian $F=ma$ business they taught us in high school. Apparently it is really handy and people really like it.

The cartpole system has two DOFs, or "generalized coordinates" -- $x$ (position), and $\theta$ (pole angle).
The system is "underactuated" because we can only externally control one -- $x$. When we push the cart, we affect its position, which in turn indirectly affects the pole angle, but we don't have an actuator attached to where the pole meets the cart (or this problem would be trivial).

The Euler-Lagrange equation lets us describe the dynamics of each generalized coordinate $q_n$:
$$ \frac{d}{dt} \frac{\partial L}{\partial \dot{q}_n} - \frac{d L}{\dot{q}_n} = \Xi_n$$
On the right hand side, we have all of the non-conservative forces (the applied force to the cart is $\Xi_x = F_x$, for example, whereas since we cannot apply any force to the angle of the pole $\Xi_\theta = 0$). On the left hand side, we have all the stuff that happens without actuation, due to gravity, masses, energy, and so on.
We solve for the equations of motion for $x$ and $\theta$ respectively.

We're done! ... or are we? We have the equations of motion, but the thing is, they were always just a means to an end. Because we ultimately want to be able to control this system, we will need to be able to solve these equations of motion. As it stands, they have all these wibbly wobbly bits and weird multiplicative terms that would make that hard. So, we have to **linearize** the system with some approximations.

The intuition here is that if you zoom far enough into any curve, it looks like a straight line. And in fact, if we approximate our wibbly wobbly sines and cosines with straight lines, we can come up with much simpler equations of motion, and easly solve for $\ddot{\theta}$ and $\ddot{x}$. That is what the last page of the lecture notes does.

### Next: what force should we apply to get the pole angle close to 0?
In the [PID](../pid/PID.ipynb) notebook, we implemented a proportional-derivative (PD) controller that had two gains $k_p$ and $k_d$ which describe how the controller should react to an error on the pole angle and a change in error between timesteps, respectively. We chose magic values for these, and thought of them separately: this one reacts to error, that one reacts to change in error.
To be formal about it, we're doing this:
$$
F = 
\begin{pmatrix}
0 & 0 & k_p & k_d
\end{pmatrix}
\begin{pmatrix}
x \\
\dot{x} \\
\theta \\
\dot{\theta}
\end{pmatrix}
= K_{PD} \vec{x}
$$
(Please note that the conventional notation actually has the gains negative and the equation as $F = Kx$, but in keeping with the fact that in [my implementation](../pid/pd_agent.py) I defined all the gains to be positive, I've written it an unconventional way.)

But what if those two 0 values weren't zero, so that we could also react to the position and velocity of the cart? And what if instead of thinking of each element in that K vector separately, we controlled all of them together? 

![](Galaxy_brain.jpg)

Turns out that is exactly what LQR does. It is a galaxy-brain generalization of the PID controller which solves for the optimal gain matrix $K$, which has shape $(m,n)$ where $m$ is the number of inputs (things you have control over, in our case 1 so $K$ is a column-vector) and $n$ is the dimension of your state / observation (in our case, 4), so that $u=Kx$ where $u$ is our control input (in this case, $F_x$).


### Deriving optimal K using quadratic cost
What do we mean by "optimal"? As always, that's incredibly subjective, but LQR assumes that our cost can be described in the form:
$$J = \sum_{t=0}^{\infty} \left( \mathbf{x}_t^T Q \mathbf{x}_t + u_t^T R u_t \right)$$

For us AI-brained code monkeys this part (minimize cost function) should be familiar. But what comes next is surprising: we can do this analytically!

A small note: the [Gymnasium cartpole implementation](https://github.com/openai/gym/blob/master/gym/envs/classic_control/cartpole.py) sites [this paper](https://coneural.org/florian/papers/05_cart_pole.pdf) as their source of the cartpole equations. Unlike the previous lecture notes, these notes assume the pole is a uniform rigid rod, not a point mass at the tip of the pole.

We take the equations for $\ddot{\theta}$ and $\ddot{x}$ and put them into matrix form to get an equation for $x_{t+1} = A x_t + B F_t$:

$$A_c = \begin{pmatrix} 0 & 1 & 0 & 0 \\ 0 & 0 & -\dfrac{m\ell g}{m_t d} & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & \dfrac{g}{d} & 0 \end{pmatrix}, \quad B_c = \begin{pmatrix} 0 \\ \dfrac{1}{m_t}\left(1 + \dfrac{m\ell}{m_t d}\right) \\ 0 \\ -\dfrac{1}{m_t d} \end{pmatrix}$$
where
$$ d \equiv \ell\left(\frac{4}{3} - \frac{m}{m_t}\right). $$

Then, by some magic I don't understand (basically doing what [this guy](https://www.mwm.im/lqr-controllers-with-python/) does), we call `solve_discrete_are` and invert some matrices to get $K$, and it works amazingly well:


In [ ]:
import sys
sys.path.append('..')
from agent import init_agent, reset_agent, get_action, update, model_name


In [2]:
# eval
from eval import eval_loop
eval_results = eval_loop(init_agent, reset_agent, get_action, model_name)

/Users/miranda/miniconda3/envs/nmm_finalproject/lib/python3.12/site-packages/gymnasium/envs/registration.py:729: UserWarning: WARN: The environment is being initialised with render_mode='none' that is not in the possible render_modes (['human', 'rgb_array']).
  logger.warn(
repeat 9: 100%|██████████| 100/100 [00:02<00:00, 45.68it/s, steps=500]

steps alive across 10 reps, 100 per rep: 500.00 ± 0.00


In [4]:
import sys
sys.path.append('..')
from agent import init_agent, reset_agent, get_action, update, model_name
from utils.record_video import record_video

record_video(
    init_agent,
    reset_agent,
    get_action,
    model_name
)

/Users/miranda/miniconda3/envs/nmm_finalproject/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /Users/miranda/code/mit/spring26/nmm/finalproject/lqr folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


recorded 500-step rollout to /LQR-episode-0.mp4
